# Notebook 03 — BART Fine-tuning
Fine-tune facebook/bart-large-cnn on DialogSum, evaluate on test set.

In [ ]:
import sys, json
sys.path.insert(0, '..')

from src.data.loader import load_dialogsum
from src.methods.bart_finetune import train, predict, save_results
from src.evaluation.visualize import plot_training_curves


## Load Data

In [ ]:
dataset = load_dialogsum(cache_dir='../data/raw/dialogsum')
print(dataset)


## Train (skip if already trained)

In [ ]:
# ~75 minutes on A5000. Set SKIP_TRAINING=True to load a pre-trained checkpoint.
SKIP_TRAINING = False

if not SKIP_TRAINING:
    trainer = train(
        dataset,
        config_path='../configs/bart_config.yaml',
        output_dir='../models/bart_finetuned',
    )
    # Plot training curves
    plot_training_curves(
        trainer.state.log_history,
        metric='rougeL',
        save_path='../results/figures/training_curves_bart.png',
        title='BART Fine-tuning — Loss & ROUGE-L',
    )


## Inference on Test Set

In [ ]:
test = dataset['test']
dialogues  = test['dialogue']
references = test['summary']

predictions = predict(
    dialogues,
    model_dir='../models/bart_finetuned',
    batch_size=8,
)
print('Sample prediction:', predictions[0])
print('Reference:        ', references[0])


## Evaluate & Save

In [ ]:
scores = save_results(predictions, references, '../results/scores/bart_results.json')
print('BART scores:', scores)
